# 01 · Exploratory Data Analysis — Trade Perfect
What in the 50K labelled history predicts a settlement failure? Every chart below drove a design decision in the engine (noted under each).

In [ ]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "backend"))
DATA = ROOT / "data"
from app.engine import EngineConfig, load_historical_csv
from app.engine.reference import load_reference_data, compute_counterparty_tiers
plt.rcParams.update({"figure.figsize": (8, 3.6), "axes.spines.top": False, "axes.spines.right": False})
hist = load_historical_csv(str(DATA / "Historical_Trade_Dataset_50K.csv"))
refs = load_reference_data(str(DATA / "Ref_Data"), hist)
y = hist.settlement_failed
print(f"{len(hist):,} historical trades | fail rate {y.mean():.1%} | {hist.trade_date.min()} -> {hist.trade_date.max()}")

## 1. Label balance
67% of historical trades failed → accuracy alone is misleading (always saying *fail* = 67%). We judge the model on AUC, precision, recall, F1 and FPR too.

In [ ]:
vc = y.map({True: "Failed", False: "Settled"}).value_counts()
fig, ax = plt.subplots(); bars = ax.bar(vc.index, vc.values, color=["#f5222d", "#52c41a"])
ax.set_title("Settlement outcome (50K history)"); ax.bar_label(bars); plt.show()

## 2. Fail rate by the main drivers
**Decision:** SSI gets the biggest rule weight (0.35), currency the second (0.15).

In [ ]:
def rate(col, labels=None):
    r = hist.groupby(col).settlement_failed.agg(["mean", "size"])
    if labels: r.index = r.index.map(labels)
    return r
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
for ax, (title, col) in zip(axes, [("SSI matched", "ssi_matched"), ("Currency matched", "currency_matched"), ("Trade type", "trade_type")]):
    r = rate(col, {True: "YES", False: "NO"} if col != "trade_type" else None)
    bars = ax.bar(r.index.astype(str), r["mean"].values, color="#1677ff"); ax.set_ylim(0, 1); ax.set_title(f"Fail rate by {title}")
    ax.bar_label(bars, labels=[f"{v:.1%}" for v in r["mean"].values])
plt.tight_layout(); plt.show()
both_ok = hist[hist.ssi_matched & hist.currency_matched].settlement_failed.mean()
print(f"SSI and currency both matched -> only {both_ok:.1%} fail")

## 3. Quantity
**Decision:** rules S-QTY-HIGH (> 5,000, +0.10) and S-QTY-LOW (< 100, +0.05). Every trade outside 100–5,000 failed.

In [ ]:
bins = pd.cut(hist.quantity, [0, 99, 1000, 2000, 3000, 4000, 5000, 1e9], labels=["<100", "100-1K", "1-2K", "2-3K", "3-4K", "4-5K", ">5K"])
r = hist.groupby(bins, observed=True).settlement_failed.mean().dropna()
fig, ax = plt.subplots(); bars = ax.bar(r.index.astype(str), r.values, color="#faad14")
ax.set_ylim(0, 1.05); ax.set_title("Fail rate by quantity bucket")
ax.bar_label(bars, labels=[f"{v:.0%}" for v in r.values]); plt.show()

## 4. Settlement timing
**Decision:** S-SD-FAR (> 15 days after **trade date**) and S-SD-WKND. Note: the spec measured "far" from today's date, which would flag 64% of the demo file wrongly — we measure from the trade date.

In [ ]:
days = (pd.to_datetime(hist.settlement_date) - pd.to_datetime(hist.trade_date)).dt.days
wk = pd.to_datetime(hist.settlement_date).dt.dayofweek >= 5
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
r = hist.groupby(pd.cut(days, [0, 2, 5, 10, 15, 100], labels=["1-2", "3-5", "6-10", "11-15", ">15"]), observed=True).settlement_failed.mean()
axes[0].bar(r.index.astype(str), r.values, color="#722ed1"); axes[0].set_title("Fail rate by days to settle"); axes[0].set_ylim(0, 1)
r2 = hist.groupby(wk).settlement_failed.mean()
axes[1].bar(["Weekday", "Weekend"], r2.values, color="#722ed1"); axes[1].set_title("Fail rate: weekend settlement"); axes[1].set_ylim(0, 1)
plt.tight_layout(); plt.show()

## 5. Counterparty risk tiers — finding Q15
Tiers come from the **count** of failures (§2.3). But every counterparty fails ≈ 66.7% of the time, and fail count tracks trade count perfectly → tiers measure **volume**, not risk. Kept per the business rule; raised with mentors.

In [ ]:
t = pd.DataFrame(compute_counterparty_tiers(hist)).T.astype({"fail_count": int, "trade_count": int, "fail_rate": float, "risk_tier": int})
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
colors = t.risk_tier.map({1: "#52c41a", 2: "#1677ff", 3: "#faad14", 4: "#f5222d"})
axes[0].scatter(t.trade_count, t.fail_count, c=colors); axes[0].set_xlabel("trades"); axes[0].set_ylabel("failed trades")
axes[0].set_title(f"Fails vs trades (corr = {t.trade_count.corr(t.fail_count):.3f})")
axes[1].bar(t.index, t.fail_rate, color=colors); axes[1].set_ylim(0.6, 0.7); axes[1].tick_params(axis="x", rotation=90, labelsize=7)
axes[1].set_title("Fail RATE per counterparty (coloured by tier)")
plt.tight_layout(); plt.show()
print(t.groupby("risk_tier").fail_rate.agg(["min", "max", "size"]))

## 6. Trader name and price — no signal
Trader fail rates sit around 66–68% → trader name is **not** a model feature. Prices are synthetic (every symbol ≈ mean 10,000) → price z-score is kept only because the spec requires it.

In [ ]:
r = hist.groupby("trader_name").settlement_failed.mean().sort_values()
ax = r.plot.barh(color="#8c8c8c", figsize=(7, 4), title="Fail rate by trader"); ax.set_xlim(0.6, 0.72); plt.show()
print(hist.groupby("symbol").price.mean().describe().round(0))

## 7. The demo file (20K) vs history — data traps and drift
**Decision:** hard gates R-000..R-008 and a PSI drift monitor.

In [ ]:
from app.engine.normalize import read_trade_csv, normalize_records
from app.engine.hard_gates import run_hard_gates
from collections import Counter
recs = read_trade_csv(str(DATA / "Trade_Dataset_20K.csv"))
g = run_hard_gates(normalize_records(recs), refs)
rej = [e for e in g.errors if e]
print(f"REJECTED {len(rej)} ->", dict(Counter(x.rule_id for e in rej for x in e)))
live = normalize_records(recs).df
cmp = pd.DataFrame({
    "history": [hist.ssi_matched.mean(), hist.currency_matched.mean(), (pd.to_datetime(hist.settlement_date).dt.dayofweek >= 5).mean(), (hist.quantity > 5000).mean()],
    "20K file": [live.ssi_matched.astype(float).mean(), live.currency_matched.astype(float).mean(),
                 (pd.to_datetime(live.settlement_date).dt.dayofweek >= 5).mean(), (live.quantity.astype(float) > 5000).mean()]},
    index=["SSI matched", "Currency matched", "Weekend settlement", "Quantity > 5000"])
cmp.plot.bar(rot=0, title="History vs 20K demo file", color=["#1677ff", "#faad14"]); plt.show(); cmp.round(3)

## Summary of findings → decisions
| Finding | Decision |
|---|---|
| SSI = NO → 97% fail | Rule weight 0.35; top model feature |
| Currency = NO → 90% fail | Rule weight 0.15 |
| Qty < 100 or > 5,000 → 100% fail | Quantity rules |
| > 15 days after trade → higher fail | S-SD-FAR measured from trade date |
| Tiers = volume, not risk | Kept per spec, raised as Q15 |
| Trader / BUY-SELL / price carry no signal | Not used or low weight |
| 704 invalid rows, weekend share 34% → 4% | Hard gates + PSI drift monitor |